# 持仓条件稀疏动作：市场拟合前的失败诊断

本轮保持原27个市场输入、初始持仓路径、费用、F和风险门，检验持仓相关的精确HOLD结构。已登记的合成正控失败，因此市场拟合被停止。本Notebook只核对原生冻结输入并通过Qlib Recorder归档合成证据。

In [1]:
from pathlib import Path
import sys, json, qlib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'SPEC.md').exists())
sys.path.insert(0,str(ROOT))
CARD=ROOT/'research/sparse_holdings_net_excess_20261003'
PLAN=json.loads((CARD/'selection_plan.json').read_text())
qlib.init(provider_uri=str(ROOT/PLAN['source']['provider_uri']),region='us')
from qlib.data import D
from qlib.workflow import R
print({'calendar_sessions':len(D.calendar(freq='day')),'as_of':PLAN['source']['as_of'],'objective_achieved':False})

[28621:MainThread](2026-10-03 05:06:24,748) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[28621:MainThread](2026-10-03 05:06:25,118) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[28621:MainThread](2026-10-03 05:06:25,118) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-worktrees/sparse-holdings-net-excess-20261003/data/qlib/published_futu_affine_20260925_v1')}


{'calendar_sessions': 4162, 'as_of': '2026-09-25', 'objective_achieved': False}


原生DatasetH重新校验冻结输入与成熟期。下一个单元格只记录失败正控，不调用市场训练或市场回测；合成行号不伪装成交易日。首次记录接收带SHA的外部证据，后续由preflight_probe_reference.json从Recorder重载，Notebook不依赖临时路径。

In [2]:
import os
from src.workflow.sparse_holdings_net_excess import record_preflight_diagnostic
summary=record_preflight_diagnostic(
    ROOT,CARD,
    probe_path=os.environ.get('SOXL_PREFLIGHT_PROBE_PATH'),
    expected_probe_sha256=os.environ.get('SOXL_PREFLIGHT_PROBE_SHA256'),
    source_receipt_path=os.environ.get('SOXL_PREFLIGHT_SOURCE_RECEIPT'))
print(json.dumps({'recorder_id':summary['recorder_id'],'scope':summary['record_type'],
                  'decision':summary['decision'],'market_fits':summary['market_model_fits'],
                  'objective_achieved':summary['objective_achieved']},ensure_ascii=False,indent=2))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[28621:MainThread](2026-10-03 05:06:26,430) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[28621:MainThread](2026-10-03 05:06:26,431) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[28621:MainThread](2026-10-03 05:06:26,431) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace-worktrees/sparse-holdings-net-excess-20261003/data/qlib/published_futu_affine_20260925_v1')}


[28621:MainThread](2026-10-03 05:06:26,530) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[28621:MainThread](2026-10-03 05:06:29,810) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name sparse_holdings_net_excess_20261003.


[28621:MainThread](2026-10-03 05:06:29,812) INFO - qlib.workflow - [exp.py:258] - Experiment 415159479695254065 starts running ...


[28621:MainThread](2026-10-03 05:06:29,867) INFO - qlib.workflow - [recorder.py:345] - Recorder a728372af45c4e678f326996db675a43 starts running under Experiment 415159479695254065 ...


[28621:MainThread](2026-10-03 05:06:29,950) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{
  "recorder_id": "a728372af45c4e678f326996db675a43",
  "scope": "failed_registered_synthetic_preflight_diagnostic",
  "decision": "aborted_before_market_fitting_failed_registered_positive_control",
  "market_fits": 0,
  "objective_achieved": false
}


In [3]:
import pandas as pd
display(pd.read_csv(CARD/'preflight_seed_summary.csv'))
print(json.dumps(summary['preflight'],ensure_ascii=False,indent=2))

,seed,ordinal_rows,saved_states,healthy,selected_epoch,retained_F,four_tail_constraints_pass,null_selected,feasible_positive_F_states,feasible_F_above_numerical_margin_states,maximum_trajectory_F,registered_positive_control_pass
0,17,504,121,True,anchor,0.0,True,True,0,0,0.126089,False
1,41,504,121,True,anchor,0.0,True,True,0,0,0.126725,False
2,71,504,121,True,anchor,0.0,True,True,0,0,0.115282,False


{
  "fixture_rows": 504,
  "chronology": "ordinal synthetic rows0..503; no market calendar dates",
  "features_sha256": "340db41ecb071c754b8f971b76068203eefb854dd48080549e3a7abe1106ddbf",
  "returns_sha256": "9327f782d854ed39a7b56104924499f34c892dce152147c852741d8b697f72b9",
  "registered_positive_control_pass": false,
  "market_data_used_for_toy": false,
  "retention_reproduction_source_sha256": "3c690d1185bbd0ff8c64c64f149320eabfbbcd79fd418a192c5fad78b54fbe8d",
  "initial_failed_probe_source_sha256": "26ed0c13087973061ada0704255dd7763b58d04afb6dbdcddbb957f4b39ef1dd",
  "financial_summary_matches_initial": true,
  "full_traces_belong_to_retention_reproduction": true,
  "witness_from_original_summary": {
    "F": 0.25546977948902533,
    "feasible": true
  },
  "witness_newly_evaluated": false,
  "F_units": "63 times mean net log excess on synthetic ordinal steps; not observed financial performance"
}


# 持仓条件稀疏动作：拟合前正控失败

本轮实现272参数的四专家稀疏动作模型，保留原27→8市场编码器，增加实际持仓分支；原模型整个策略类及初始持仓路径可嵌入。另用独立版本修正费用后持仓状态导数，并明确标量费用拐点的梯度约定。**数学与数据合同通过，但已登记的优化能力正控失败，因此没有进行市场拟合或市场回测，不晋升。稳定净超额目标仍未完成。**

## 实际运行与结果

原生Qlib复核覆盖3,956行原样本、4,162个日历点、14个冻结预测模型和21个映射训练折；未重新拟合预测器。完整模型的持仓传递、精确HOLD、逐日异常预测回退和共享持仓集合接口通过检查。梯度与独立全路径/截断路径计算误差均低于8e−16，有限差分误差低于2.2e−9；这些是工程与数学证据，不是投资收益证据。

优化器正控是预先固定的504行人工序列：第一特征交替+1/−1，其他26列为0；人工SOXX收益为0.002+0.008x，人工SOXL为其三倍。它没有真实交易日期。每个种子仍使用120次更新、两档成本和四项原风险门。

|种子|正F状态数|同时风险可行的正F状态|最终保留|观察到的停滞|
|---|---:|---:|---|---|
|17|52|0|SOXX，F=0|第52轮进入全部纯SOXX，之后68次更新梯度为0|
|41|53|0|SOXX，F=0|第53轮进入全部纯SOXX，之后67次更新梯度为0|
|71|45|0|SOXX，F=0|最终候选只有SOXX/HOLD；可行但F=−0.000298819|

F是登记的63倍平均净对数超额目标，不能当作收益百分比。人工序列上存在已经独立验证的可达方案：F约0.25547、两成本四项风险约束均通过。因此失败指向本次有限预算的优化过程，不能推出模型类不存在可行改进，更不能推出真实市场没有信号。

17/41的稀疏输出进入单一SOXX动作后，对分数的局部梯度归零。71中的SOXX和HOLD随着自身持仓接近SOXX而产生几乎相同的实际动作，剩余梯度主要涉及初始入场，未重新激活SOXL或现金。这解释了为何仅增加相同局部梯度更新未必能解决问题；本轮没有增加迭代、修改学习率或寻找更容易通过的合成序列。

独立审计重建363个检查点、726条成本路径、365,904个决策，F最大差约1.40e−14，原始风险差约4.41e−14个百分点，对偶更新完全一致。没有发现会解释失败的现金账、路径或对偶算术错误。

## 证据身份与局限

初次正控及部分数学/运行时间审查使用模型源码`26ed0c13…`；随后仅将诊断字段`reported_trade_count_gt_1e12`更名为`reported_trade_count_gt_1e_minus12`，源码变为`3c690d11…`。完整轨迹来自后者的一次固定参数保留重放。两份源码、原始摘要、身份收据和完整保留轨迹均归档；没有声称原先未保存的全部轨迹已被逐状态比对。

本卡没有新的市场收益、行情阶段或8月3日窗口结果。最近完成的市场检验仍是[上一轮净超额研究](../constrained_net_excess_20261003/conclusion.md)，它也未通过稳定超额门。当前记录FINISHED仅表示失败诊断已归档，不表示模型合格。原SIM、生产策略和真实账户均未改变。

## 下一项待登记检验

下一步拟从上一轮已验证的可行训练模型出发，精确嵌入后固定其市场编码器，对输出与持仓系数使用有约束的有限参数搜索。它能检验局部梯度无法离开的区域；仍可能失败。将要求相对父模型取得实际增益并满足同一风险门，避免仅继承父模型就宣称优化成功。该方法尚未实现或运行，不能当作本轮成果；源码、预算及合成增益门将在运行前另行登记。

Qlib诊断Recorder：**a728372af45c4e678f326996db675a43**。主Notebook已执行；77项针对性检查通过。全套测试为1,182通过、139项既有失败，规定Qlib测试为8通过、61项既有失败；失败ID与上一轮完全相同，没有新增失败。完整核对见verification.json。大轨迹、原始源码及输入身份留在Recorder，卡根目录只保留小型摘要与结论。
